# QMCPy logo

Generate a lightweight scatter-plot logo from a two-dimensional randomized digital net, deterministic digital net, or lattice transformed to a zero-mean multivariate normal distribution with covariance `[[2, 1], [1, 3]]`. Leave exactly one sampler line uncommented; the output filename follows the choice automatically. The deterministic net starts at index 1 because its origin maps to non-finite Gaussian coordinates.

In [ ]:
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
from qmcpy import DigitalNetB2, Gaussian, Lattice

n = 128
seed = 7
covariance = np.array([[2.0, 1.0], [1.0, 3.0]])
assert np.all(np.linalg.eigvalsh(covariance) > 0)

# Choose one sampler by leaving exactly one of these lines uncommented.
# sampler, output_suffix, start_index = DigitalNetB2(dimension=2, seed=seed), "_net", 0
# sampler, output_suffix, start_index = Lattice(dimension=2, seed=seed), "_lattice", 0
sampler, output_suffix, start_index = DigitalNetB2(dimension=2, randomize=False, order="GRAY"), "_det_net", 1

# Work whether Jupyter starts in the repository root or in demos/.
start = Path.cwd().resolve()
repo_root = None
for path in (start, *start.parents):
    if (path / "pyproject.toml").exists() and (path / "qmcpy").is_dir():
        repo_root = path
        break
if repo_root is None:
    raise RuntimeError(
        "Could not locate repository root (expected pyproject.toml and qmcpy/). Run this notebook from within the QMCSoftware repository."
    )
current_logo = repo_root / "docs" / "assets" / "logos" / "qmcpy_logo.png"
target = repo_root / "docs" / "logos" / f"qmcpy_logo{output_suffix}.png"


In [ ]:
# Read the dominant opaque RGB color from the present logo.
rgba = mpimg.imread(current_logo)
opaque_rgb = rgba[..., :3][rgba[..., 3] > 0.99]
rgb, counts = np.unique(
    np.round(opaque_rgb * 255).astype(np.uint8), axis=0, return_counts=True
)
logo_color = rgb[counts.argmax()] / 255

normal = Gaussian(
    sampler,
    mean=np.zeros(2),
    covariance=covariance,
)
points = normal(n_min=start_index, n_max=start_index + n, warn=False)


In [ ]:
target.parent.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 8), facecolor="none")
ax.scatter(points[:, 0], points[:, 1], s=150, color=logo_color, edgecolors="none")
ax.set_aspect("equal")
center = (points.min(axis=0) + points.max(axis=0)) / 2
radius = np.ptp(points, axis=0).max() * 0.54
ax.set_xlim(center[0] - radius, center[0] + radius)
ax.set_ylim(center[1] - radius, center[1] + radius)
ax.axis("off")
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
fig.savefig(target, dpi=196, transparent=True)
plt.show()

print(f"Wrote:     {target.relative_to(repo_root)}")